In [ ]:
import numpy as np
import xarray as xr 
import pandas as pd
import glob
import os

import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, detrend

from matplotlib.ticker import LogLocator, FuncFormatter

import sys
sys.path.append('../..//')
from utils_mitgcm import open_mitgcm_ds_from_config
from utils_signal_processing import *

# Load data and select ZZ

In [ ]:
lake = 'geneva'
model = f'{lake}_2025'
mitgcm_config, ds = open_mitgcm_ds_from_config('../../config.json', model)

In [ ]:
folder_path = os.path.dirname(mitgcm_config['datapath'])
output_folder = os.path.join(folder_path, "seiche_analysis")
os.makedirs(output_folder, exist_ok=True)

In [ ]:
grid_resolution = 100
ds['YC'] = np.arange(1, len(ds['YC'])+1) * grid_resolution - grid_resolution/2
ds['XC'] = np.arange(1, len(ds['XC'])+1) * grid_resolution - grid_resolution/2
ds['YG'] = np.arange(0, len(ds['YG'])) * grid_resolution
ds['XG'] = np.arange(0, len(ds['XG'])) * grid_resolution

In [ ]:
omega = 7.2921e-5  # Earth's angular velocity (rad/s)
lat_rad = np.deg2rad(46.45)# Latitude of Lake Geneva (degrees)
f = 2 * omega * np.sin(lat_rad) # Coriolis parameter
f0 = abs(f/(2*np.pi)) # conversion from rad/s to 1/s

## Select sub XY for mean freq. Spectrum

In [ ]:
start_date = np.datetime64('2025-08-01T00:00:00', 'ns')
end_date = np.datetime64('2025-09-01T00:00:00', 'ns')

In [ ]:
zz = 10
xx1 = 49850 # Lucerne: 13700, Geneva: 49850, Neuchatel: 10000
yy1 = 18100 # Lucerne: 9100, Geneva: 18100, Neuchatel: 4000
xcells = 4
ycells = 4
xx2 = xx1+(grid_resolution*xcells)
yy2 = yy1+(grid_resolution*ycells)

In [ ]:
fig, ax = plt.subplots(1, figsize=(15, 8))
ds.VVEL.isel(time=24, Z=zz).plot()
plt.scatter(xx1,yy1, marker=".", color="k")
plt.scatter(xx1,yy2, marker = '.', color="k")
plt.scatter(xx2,yy1, marker=".", color="k")
plt.scatter(xx2,yy2, marker = '.', color="k")

plt.grid()

ds['THETA'].sel(XC=xx1, YC=yy1, method='nearest').to_netcdf(os.path.join(output_folder, 'theta_lexplore.nc'))

In [ ]:
u = ds.UVEL.isel(Z=zz)\
          .sel(XG=slice(xx1,xx2), YC=slice(yy1,yy2))\
          .sel(time=slice(start_date, end_date))\
          .mean(['XG','YC'])\
          .load()

v = ds.VVEL.isel(Z=zz)\
          .sel(XC=slice(xx1,xx2), YG=slice(yy1,yy2))\
          .sel(time=slice(start_date, end_date))\
          .mean(['XC','YG'])\
          .load()

w = ds.WVEL.isel(Zl=zz)\
          .sel(XC=slice(xx1,xx2), YC=slice(yy1,yy2))\
          .sel(time=slice(start_date, end_date))\
          .mean(['XC','YC'])\
          .load()

u = u.chunk({'time':-1})
v = v.chunk({'time':-1})
w = w.chunk({'time':-1})

u.load()
v.load()
w.load()

# Compute freq. spectrum

## Compute freq. spectrum

In [ ]:
u.plot()

In [ ]:
v.plot()

In [ ]:
l_seg = len(u.time)
u_fft = xr_compute_meanfft(u, seg_length=l_seg)
v_fft = xr_compute_meanfft(v, seg_length=l_seg)
w_fft = xr_compute_meanfft(w, seg_length=l_seg)

In [ ]:
u_fft_mean = u_fft#.mean(dim=['XG','YC'])
v_fft_mean = v_fft#.mean(dim=['XC','YG'])
w_fft_mean = w_fft#.mean(dim=['XC','YC'])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, LogFormatter
from scipy.signal.windows import dpss

def rotary_psd(u, v, f, dt=1.0, NW=3, K=5, plot=True):
    """
    Compute rotary PSD for 2D vector time series (u, v) with multitaper.
    Works for odd or even length signals.
    """
    z = np.asarray(u) + 1j*np.asarray(v)
    n = len(z)

    # Generate DPSS tapers
    tapers = dpss(n, NW, Kmax=K)

    # FFT length
    nfft = n
    pos_len = nfft // 2 if n % 2 == 0 else nfft // 2 + 1  # number of positive freqs

    # Initialize spectra
    Spp = np.zeros(pos_len)
    Snn = np.zeros(pos_len)

    # Loop over tapers
    for k in range(K):
        tapered = z * tapers[k]
        Z = np.fft.fft(tapered)/n
        Spp += np.abs(Z[:pos_len])**2
        Snn += np.abs(Z[-pos_len:][::-1])**2  # flip negative freqs to match pos axis

    Spp /= K
    Snn /= K

    # Frequency axis (positive)
    freqs_pos = np.fft.fftfreq(n, dt)[:pos_len]

    # Plot
    if plot:
        fig, ax = plt.subplots(1,1, figsize=(8,5))

        ax.loglog(freqs_pos, Spp, label='Positive (CW)')
        ax.loglog(freqs_pos, Snn, label='Negative (CCW)')

        ax.vlines(f, 0, Spp.max(), linestyle='--', color='k', label='Coriolis Frequency')

        ax.xaxis.set_major_locator(LogLocator(base=10, subs=np.arange(1,10,3)))

        # Create a secondary x-axis for period in hours
        secax = ax.secondary_xaxis('top')
        secax.set_xscale('log')
        secax.set_xlabel('Period (hours)')
        secax.set_xticks(ax.get_xticks())
        secax.set_xticklabels([f'{int(x):d}' for x in compute_fft_period(ax.get_xticks()) / 3600])


        plt.xlabel('Frequency [Hz]')
        plt.ylabel('PSD [unit^2/Hz]')
        plt.title('Rotary Power Spectral Density')
        plt.legend()
        plt.grid(True, which='both', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.show()

    return freqs_pos, Spp, Snn

In [ ]:
freqs, Spp, Snn = rotary_psd(u, v, f0, dt=3600, NW=3, K=5)

## Plot freq. spectrum

In [ ]:
fig,ax = plot_freq_spectrum(u_fft_mean, 'U', depth=zz, l_segm=l_seg, y_lim_min=1e-10, x_lim_min=1e-6, fontsize=10)
ax.axvline(x = f0, linestyle="--", color="k",label="f0")
#fig.savefig(os.path.join(output_folder, 'freq_spectrum.png'))

In [ ]:
fig,ax = plot_freq_spectrum(v_fft_mean, 'V', depth=zz, l_segm=l_seg, y_lim_min=1e-10, x_lim_min=0.01e-4, fontsize=10)
ax.axvline(x = f0, linestyle="--", color="k",label="f0")

# Filtering in Spectral space  

## Defining cutoffs 
- Inertial period is at 16.4 hrs

In [ ]:
cutoff1_hr = 45
cutoff2_hr = 30

cutoff1 = 1/(cutoff1_hr * 3600)
cutoff2 = 1/(cutoff2_hr * 3600)

In [ ]:
fig,ax = plot_freq_spectrum(u_fft_mean, 'U', depth=zz, l_segm=l_seg, y_lim_min=1e-11, x_lim_min=0.01e-4, fontsize=10)
ax.axvline(x = cutoff1, linestyle="--", color="k",label="cutoff1")
ax.axvline(x = cutoff2, linestyle="--", color="k",label="cutoff2")
ax.legend()
fig.savefig(os.path.join(output_folder, f'freq_spectrum_with_cutoffs_{cutoff2_hr}_{cutoff1_hr}h.png'))

In [ ]:
path_cutoff_folder = os.path.join(output_folder, f'{cutoff2_hr}_{cutoff1_hr}h')
os.makedirs(path_cutoff_folder, exist_ok=True)

## Low-pass filtering (slow motions)

### Filtering to low freq. motions

In [ ]:
ulow = filter_signal_xarray(u, btype='lowpass', time_dim='time', dt=3600, period_cutoff_high=(cutoff1_hr*3600), order=5)
vlow = filter_signal_xarray(v, btype='lowpass', time_dim='time', dt=3600, period_cutoff_high=(cutoff1_hr*3600), order=5)
wlow = filter_signal_xarray(w, btype='lowpass', time_dim='time', dt=3600, period_cutoff_high=(cutoff1_hr*3600), order=5)

### Plotting low freq. motions 

In [ ]:
i_time_to_plot = 48

vmax_low = 7.5e-2
ulow.plot()
plt.grid()

## Bandpass filtering (seiches)

### Filtering to seiches

In [ ]:
useiche = filter_signal_xarray(u, btype='bandpass', time_dim='time', dt=3600, period_cutoff_low=(cutoff1_hr*3600), period_cutoff_high=(cutoff2_hr*3600), order=5)
vseiche = filter_signal_xarray(v, btype='bandpass', time_dim='time', dt=3600, period_cutoff_low=(cutoff1_hr*3600), period_cutoff_high=(cutoff2_hr*3600), order=5)
wseiche = filter_signal_xarray(w, btype='bandpass', time_dim='time', dt=3600, period_cutoff_low=(cutoff1_hr*3600), period_cutoff_high=(cutoff2_hr*3600), order=5)

### Plotting seiches 

In [ ]:
i_time_to_plot += 1
print(i_time_to_plot)
vmax_seiche = 7.5e-2
useiche.plot()
plt.grid()

## High-pass filtering (high frq. waves)

### Filtering to high freq. waves 

In [ ]:
uhigh = filter_signal_xarray(u, btype='highpass', time_dim='time', dt=3600, period_cutoff_low=(cutoff2_hr*3600), order=5)
vhigh = filter_signal_xarray(v, btype='highpass', time_dim='time', dt=3600, period_cutoff_low=(cutoff2_hr*3600), order=5)
whigh = filter_signal_xarray(w, btype='highpass', time_dim='time', dt=3600, period_cutoff_low=(cutoff2_hr*3600), order=5)

### Plot high freq. internal waves

In [ ]:
vmax_high = 7.5e-3
vhigh.plot()
plt.grid()

## Comparing time series

In [ ]:
def plot_comparison_timeseries(vel, low, seiche, high, name_var):
    fig, ax = plt.subplots(2, 1, figsize=(18, 8))

    # bottom plot
    vel.plot(ax=ax[1], label="Original")
    (low + seiche + high).plot( ax=ax[1], label="low freq + seiche + high freq")

    # upper plot
    low.plot(ax=ax[0], label="low freq.")
    seiche.plot(ax=ax[0], label="seiche")
    high.plot(ax=ax[0], label="high freq.")

    for ax_i in (ax[1], ax[0]):
        ax_i.grid()
        ax_i.set_xlabel('')
        ax_i.set_title('')
        ax_i.legend(loc='upper right')
        ax_i.set_ylabel(f'{name_var} (m/s)')

    return fig, ax


In [ ]:
fig, ax = plot_comparison_timeseries(u, ulow, useiche, uhigh, 'U')
ax[0].set_title(f'U - Cutoff: {cutoff2_hr}-{cutoff1_hr}h')
plt.savefig(os.path.join(path_cutoff_folder, f'plot_filtered_timeseries_{cutoff2_hr}-{cutoff1_hr}h_U.png'))

In [ ]:
fig, ax = plot_comparison_timeseries(v, vlow, vseiche, vhigh, 'V')
ax[0].set_title(f'V - Cutoff: {cutoff2_hr}-{cutoff1_hr}h')
plt.savefig(os.path.join(path_cutoff_folder, f'plot_filtered_timeseries_{cutoff2_hr}-{cutoff1_hr}h_V.png'))

In [ ]:
fig, ax = plot_comparison_timeseries(w, wlow, wseiche, whigh, 'W')
ax[0].set_title(f'W - Cutoff: {cutoff2_hr}-{cutoff1_hr}h')
plt.savefig(os.path.join(path_cutoff_folder, f'plot_filtered_timeseries_{cutoff2_hr}-{cutoff1_hr}h_W.png'))